### Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import ast  # For safely evaluating string representations of lists
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# 1. Data Loading

In [2]:
# Load the four datasets from their respective CSV files.
try:
    books_df = pd.read_csv('data/books_formatted.csv')
    games_df = pd.read_csv('data/games_formatted.csv')
    movies_df = pd.read_csv('data/movies_formatted.csv')
    shows_df = pd.read_csv('data/shows_formatted.csv')
    print("All datasets loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading datasets: {e}. Please ensure all CSV files are in the correct directory.")
    # Exit or handle error appropriately in a real script
    exit()


All datasets loaded successfully.


# 2. Data Cleaning and Unification

In [4]:
# This section contains functions to standardize each dataframe to the unified schema.
def clean_books(df):
    """Processes the books dataframe to match the unified schema."""
    df['media_type'] = 'Book'
    df['id'] = 'book_' + df['id'].astype(str)
    # Genres are a single string, so we wrap it in a list
    df['genres'] = df['genres'].apply(lambda x: [x.lower()] if isinstance(x, str) else [])
    # Books dataset does not have a creator/company column, so we add an empty list
    df['creator'] = [[] for _ in range(len(df))]
    df.rename(columns={'overview': 'overview'}, inplace=True)
    df.dropna(subset=['name', 'overview'], inplace=True)
    return df[['id', 'name', 'media_type', 'overview', 'genres', 'creator']]

def clean_games(df):
    """Processes the games dataframe to match the unified schema."""
    df['media_type'] = 'Game'
    df['id'] = 'game_' + df['id'].astype(str)
    # The 'genres' and 'company' columns are string representations of lists. Use ast.literal_eval.
    def parse_list_string(s):
        try:
            return [item.strip().lower() for item in ast.literal_eval(s)]
        except (ValueError, SyntaxError):
            return []
    df['genres'] = df['genres'].apply(parse_list_string)
    df.rename(columns={'company': 'creator'}, inplace=True)
    df['creator'] = df['creator'].apply(parse_list_string)
    df.dropna(subset=['name', 'overview'], inplace=True)
    return df[['id', 'name', 'media_type', 'overview', 'genres', 'creator']]

def clean_movies(df):
    """Processes the movies dataframe to match the unified schema."""
    df['media_type'] = 'Movie'
    df['id'] = 'movie_' + df['id'].astype(str)
    # Genres are a single string, so we wrap it in a list
    df['genres'] = df['genres'].apply(lambda x: [x.lower()] if isinstance(x, str) else [])
    # Movies dataset does not have a creator/company column
    df['creator'] = [[] for _ in range(len(df))]
    df.dropna(subset=['name', 'overview'], inplace=True)
    return df[['id', 'name', 'media_type', 'overview', 'genres', 'creator']]

def clean_shows(df):
    """Processes the shows dataframe to match the unified schema."""
    df['media_type'] = 'Show'
    df['id'] = 'show_' + df['id'].astype(str)
    # Genres can have commas and ampersands
    def parse_show_genres(s):
        if not isinstance(s, str):
            return []
        s = s.replace('&', ',')
        return [genre.strip().lower() for genre in s.split(',')]
    df['genres'] = df['genres'].apply(parse_show_genres)
    df.rename(columns={'company': 'creator'}, inplace=True)
    # Creator might be a single string
    df['creator'] = df['creator'].apply(lambda x: [x.strip().lower()] if isinstance(x, str) else [])
    df.dropna(subset=['name', 'overview'], inplace=True)
    return df[['id', 'name', 'media_type', 'overview', 'genres', 'creator']]


In [5]:
# Apply the cleaning functions
books_clean = clean_books(books_df)
games_clean = clean_games(games_df)
movies_clean = clean_movies(movies_df)
shows_clean = clean_shows(shows_df)

In [6]:
books_clean.head()

,id,name,media_type,overview,genres,creator
0,book_0,Drowned Wednesday,Book,Drowned Wednesday is the first Trustee among ...,[fantasy],[]
1,book_1,The Lost Hero,Book,"As the book opens, Jason awakens on a school ...",[fantasy],[]
2,book_2,The Eyes of the Overworld,Book,Cugel is easily persuaded by the merchant Fia...,[fantasy],[]
3,book_3,Magic's Promise,Book,The book opens with Herald-Mage Vanyel return...,[fantasy],[]
4,book_4,Taran Wanderer,Book,Taran and Gurgi have returned to Caer Dallben...,[fantasy],[]


In [7]:
games_clean.head()

,id,name,media_type,overview,genres,creator
0,game_0,Elden Ring,Game,"Elden Ring is a fantasy, action and open world...","[adventure, rpg]","[bandai namco entertainment, fromsoftware]"
1,game_1,Hades,Game,A rogue-lite hack and slash dungeon crawler in...,"[adventure, brawler, indie, rpg]",[supergiant games]
2,game_2,The Legend of Zelda: Breath of the Wild,Game,The Legend of Zelda: Breath of the Wild is the...,"[adventure, rpg]","[nintendo, nintendo epd production group no. 3]"
3,game_3,Undertale,Game,"A small child falls into the Underground, wher...","[adventure, indie, rpg, turn based strategy]","[tobyfox, 8-4]"
4,game_4,Hollow Knight,Game,A 2D metroidvania with an emphasis on close co...,"[adventure, indie, platform]",[team cherry]


In [8]:
movies_clean.head()

,id,name,media_type,overview,genres,creator
0,movie_1,The Shawshank Redemption,Movie,Two imprisoned men bond over a number of years...,[drama],[]
1,movie_2,The Godfather,Movie,An organized crime dynasty's aging patriarch t...,"[crime, drama]",[]
2,movie_3,The Dark Knight,Movie,When the menace known as the Joker wreaks havo...,"[action, crime, drama]",[]
3,movie_4,The Godfather: Part II,Movie,The early life and career of Vito Corleone in ...,"[crime, drama]",[]
4,movie_5,12 Angry Men,Movie,A jury holdout attempts to prevent a miscarria...,"[crime, drama]",[]


In [9]:
shows_clean.head()

,id,name,media_type,overview,genres,creator
0,show_1399,Game of Thrones,Show,Seven noble families fight for control of the ...,"[sci-fi, fantasy, drama, action, adventure]",[hbo]
1,show_71446,Money Heist,Show,"To carry out the biggest heist in history, a m...","[crime, drama]","[netflix, antena 3]"
2,show_66732,Stranger Things,Show,"When a young boy vanishes, a small town uncove...","[drama, sci-fi, fantasy, mystery]",[netflix]
3,show_1402,The Walking Dead,Show,Sheriff's deputy Rick Grimes awakens from a co...,"[action, adventure, drama, sci-fi, fantasy]",[amc]
4,show_63174,Lucifer,Show,"Bored and unhappy as the Lord of Hell, Lucifer...","[crime, sci-fi, fantasy]","[fox, netflix]"


In [13]:
# Concatenate all cleaned dataframes into one master dataframe
master_df = pd.concat([books_clean, games_clean, movies_clean, shows_clean], ignore_index=True)
# drop column creator
master_df.drop(columns=['creator'], inplace=True)
master_df.head()

,id,name,media_type,overview,genres
0,book_0,Drowned Wednesday,Book,Drowned Wednesday is the first Trustee among ...,[fantasy]
1,book_1,The Lost Hero,Book,"As the book opens, Jason awakens on a school ...",[fantasy]
2,book_2,The Eyes of the Overworld,Book,Cugel is easily persuaded by the merchant Fia...,[fantasy]
3,book_3,Magic's Promise,Book,The book opens with Herald-Mage Vanyel return...,[fantasy]
4,book_4,Taran Wanderer,Book,Taran and Gurgi have returned to Caer Dallben...,[fantasy]


# 3. Feature Engineering: Creating the "Feature Soup"

In [15]:
# function to remove spaces from list elements to create single tokens
def clean_list_elements(lst):
    if not isinstance(lst, list):
        return []
    return [str(i).replace(" ", "").lower() for i in lst]

master_df['genres'] = master_df['genres'].apply(clean_list_elements)

# Combine all text features into a single string for vectorization
def create_feature_soup(x):
    # Join list elements with spaces, then join all features
    genres_str = ' '.join(x['genres'])
    # Ensure overview is a string
    overview_str = str(x['overview']).lower()
    return f"{overview_str} {genres_str}"

master_df['feature_soup'] = master_df.apply(create_feature_soup, axis=1)

print("Feature soup created. Example:")
print(master_df['feature_soup'].head())  # Added index [0] to print first row


Feature soup created. Example:
0     drowned wednesday is the first trustee among ...
1     as the book opens, jason awakens on a school ...
2     cugel is easily persuaded by the merchant fia...
3     the book opens with herald-mage vanyel return...
4     taran and gurgi have returned to caer dallben...
Name: feature_soup, dtype: object


# 4. TF-IDF Vectorization

In [16]:
# Initialize the TF-IDF Vectorizer
# We remove English stop words and limit the vocabulary to the top 5000 features
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit and transform the feature soup to create the TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(master_df['feature_soup'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (100501, 5000)


# 5. Cosine Similarity Calculation

In [17]:
# To avoid memory errors with a large similarity matrix, we compute it in chunks.
chunk_size = 500  # Adjust this based on your available RAM
num_rows = tfidf_matrix.shape[0]
num_chunks = int(np.ceil(num_rows / chunk_size))

# Directory to save similarity matrix chunks
import os, json
output_dir = 'similarity_chunks'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Initialize chunk index
chunk_metadata = {
    'format': 'npz',
    'index': []
}

from scipy import sparse
import gc

current_row = 0
for i in range(num_chunks):
    start_row = i * chunk_size
    end_row = min((i + 1) * chunk_size, num_rows)
    
    print(f"Computing chunk {i+1}/{num_chunks} (rows {start_row} to {end_row})...")
    
    # Compute similarity and convert to sparse immediately
    chunk_similarity = cosine_similarity(
        tfidf_matrix[start_row:end_row], 
        tfidf_matrix,
        dense_output=False
    )
    
    # Save as sparse npz
    chunk_filename = f'similarity_chunk_{i:03d}.npz'
    chunk_path = os.path.join(output_dir, chunk_filename)
    sparse.save_npz(chunk_path, chunk_similarity)
    
    # Update index
    chunk_metadata['index'].append({
        'file': chunk_filename,
        'start_row': int(start_row),
        'end_row': int(end_row)
    })
    
    # Free memory
    del chunk_similarity
    gc.collect()
    
    print(f"Chunk {i+1} saved to {chunk_path}")

# Save chunk index
with open(os.path.join(output_dir, 'chunk_index.json'), 'w') as f:
    json.dump(chunk_metadata, f, indent=2)

print("All similarity matrix chunks have been computed and saved.")


Computing chunk 1/202 (rows 0 to 500)...
Chunk 1 saved to similarity_chunks/similarity_chunk_000.npz
Computing chunk 2/202 (rows 500 to 1000)...
Chunk 2 saved to similarity_chunks/similarity_chunk_001.npz
Computing chunk 3/202 (rows 1000 to 1500)...
Chunk 3 saved to similarity_chunks/similarity_chunk_002.npz
Computing chunk 4/202 (rows 1500 to 2000)...
Chunk 4 saved to similarity_chunks/similarity_chunk_003.npz
Computing chunk 5/202 (rows 2000 to 2500)...
Chunk 5 saved to similarity_chunks/similarity_chunk_004.npz
Computing chunk 6/202 (rows 2500 to 3000)...
Chunk 6 saved to similarity_chunks/similarity_chunk_005.npz
Computing chunk 7/202 (rows 3000 to 3500)...
Chunk 7 saved to similarity_chunks/similarity_chunk_006.npz
Computing chunk 8/202 (rows 3500 to 4000)...
Chunk 8 saved to similarity_chunks/similarity_chunk_007.npz
Computing chunk 9/202 (rows 4000 to 4500)...
Chunk 9 saved to similarity_chunks/similarity_chunk_008.npz
Computing chunk 10/202 (rows 4500 to 5000)...
Chunk 10 saved

# 6. Saving Artifacts

In [18]:
import traceback

try:
    import os, gc, glob, psutil
    from pathlib import Path

    def get_memory_usage():
        """Get current memory usage in MB"""
        process = psutil.Process()
        return process.memory_info().rss / (1024 * 1024)

    def save_with_memory_check(obj, filepath, threshold_mb=6000):  # 6GB threshold
        current_memory = get_memory_usage()
        if current_memory > threshold_mb:
            gc.collect()  # Force garbage collection
            print(f"⚠️  High memory usage: {current_memory:.0f}MB")
        
        print(f"Saving to {filepath}... ", end='', flush=True)
        with open(filepath, 'wb') as f:
            pickle.dump(obj, f)
        print("Done")

    # Create output directory
    output_dir = Path('similarity_chunks')
    output_dir.mkdir(exist_ok=True)

    print(f"Initial memory usage: {get_memory_usage():.0f}MB")
    
    # 1. Save master dataframe in chunks if too large
    if len(master_df) > 100000:  # arbitrary threshold
        master_df_chunks = np.array_split(master_df, 2)
        for i, chunk in enumerate(master_df_chunks):
            save_with_memory_check(chunk, f'master_df_part_{i}.pkl')
    else:
        save_with_memory_check(master_df, 'master_df.pkl')

    # 2. Save TF-IDF vectorizer
    save_with_memory_check(tfidf, 'tfidf_vectorizer.pkl')

    # 3. Verify saved chunks with memory-efficient iteration
    print("\nVerifying saved artifacts:")
    npz_files = sorted(output_dir.glob('*.npz'))
    chunk_count = sum(1 for _ in npz_files)  # Memory efficient counting
    
    if chunk_count > 0:
        total_size = sum(f.stat().st_size for f in output_dir.glob('*.npz')) / (1024 * 1024)
        print(f"✓ Found {chunk_count} similarity chunks (.npz)")
        print(f"✓ Total chunks size: {total_size:.1f} MB")
        
        # Verify chunk index
        index_path = output_dir / 'chunk_index.json'
        if index_path.exists():
            with open(index_path) as f:
                chunk_metadata = json.load(f)
            print(f"✓ Chunk index found with {len(chunk_metadata['index'])} entries")
        else:
            print("⚠️  Warning: chunk_index.json not found")
    else:
        print("⚠️  Warning: No .npz chunk files found")

    # 4. Aggressive memory cleanup
    large_objects = ['tfidf_matrix', 'chunk_similarity', 'master_df', 'tfidf']
    for obj in large_objects:
        if obj in globals():
            del globals()[obj]
    
    gc.collect()
    print(f"\nFinal memory usage: {get_memory_usage():.0f}MB")
    print(f"Files saved in: {os.path.abspath('.')}")

except Exception as e:
    print(f"\n❌ Error saving artifacts: {str(e)}")
    traceback.print_exc()

Initial memory usage: 38MB


/Users/harshadmp/Folders/academic/education/202306-202506 mca ignou/semester 4/MCSP-232 Project/project/code2/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Saving to master_df_part_0.pkl... Done
Saving to master_df_part_1.pkl... Done
Saving to tfidf_vectorizer.pkl... Done

Verifying saved artifacts:
✓ Found 202 similarity chunks (.npz)
✓ Total chunks size: 36030.0 MB
✓ Chunk index found with 202 entries

Final memory usage: 413MB
Files saved in: /Users/harshadmp/Folders/academic/education/202306-202506 mca ignou/semester 4/MCSP-232 Project/project/code2
